In [1]:
pip install --upgrade --no-cache-dir boto3>=1.38.0 botocore>=1.38.0

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\afons\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
import boto3
print(boto3.__version__)

1.43.54


In [44]:
client = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

def add_user_message(messages,text):

    user_message = {
        "role": "user",
        "content": [
            { "text": text}
        ]
    }
    messages.append(user_message)


def add_assistant_message(messages,text):

    assistant_message = {
        "role": "assistant",
        "content": [
            { "text": text}
        ]
    }
    messages.append(assistant_message)


def chat(messages, system=None,temperature=1.0, stop_sequences=[]):
    params = {"modelId": model_id, "messages": messages,"inferenceConfig":{"temperature": temperature,"stopSequences": stop_sequences}}

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

#### Nem o bedrock nem o claude guardam as mensagens passadas, pelo que tem que ser feito manualmente

In [ ]:
messages = []

add_user_message(messages,"What's 1+1?")

answer = chat(messages)

add_assistant_message(messages,answer)

add_user_message(messages,"And 3 more added to that?")

answer = chat(messages)

answer

### Exercicio Chat bot

In [ ]:
messages = []

while True:

    user_input = input("> ")
    print(f"> {user_input}")


    add_user_message(messages,user_input)

    answer = chat(messages)

    add_assistant_message(messages,answer)

    print(answer)
    print("-"*3)


### System Prompt -> Fazer só com que fale algo especifico

In [30]:
# def chat(messagess):
#     system_prompt = """
#     You are an AWS cloud support specialist. Your job is to answer user queries related to cloud hosting services on AWS.
# """

#     response = client.converse(
#     modelId = model_id,
#     messages = messagess,
#     system=[{"text":system_prompt}])

#     return response["output"]["message"]["content"][0]["text"]

messages = []

add_user_message(messages,"How do i setup a postgres database?")
text = chat(messages,"You are an AWS support specialist")

print(text)

# Setting Up a PostgreSQL Database

Here are the main approaches to set up PostgreSQL:

## **1. Local Installation**

### On Ubuntu/Debian:
```bash
sudo apt update
sudo apt install postgresql postgresql-contrib
sudo systemctl start postgresql
sudo systemctl enable postgresql
```

### On macOS:
```bash
brew install postgresql
brew services start postgresql
```

### On Windows:
Download the installer from [postgresql.org](https://www.postgresql.org/download/windows/)

## **2. Initial Configuration**

After installation:

```bash
# Switch to postgres user
sudo -i -u postgres

# Access PostgreSQL prompt
psql

# Create a new database
CREATE DATABASE mydb;

# Create a new user
CREATE USER myuser WITH PASSWORD 'mypassword';

# Grant privileges
GRANT ALL PRIVILEGES ON DATABASE mydb TO myuser;
```

## **3. AWS RDS PostgreSQL (Recommended for Production)**

1. **Go to RDS Console** → Create Database
2. **Choose PostgreSQL** engine
3. **Select template** (Production, Dev/Test, or Free Tier)
4. **

### Exercise

In [33]:
messages = []

add_user_message(messages,"Write a function that checks a string for ducplicate characters")
text = chat(messages,"You are a python engineer who writes very concise code.")

print(text)

```python
def has_duplicates(s: str) -> bool:
    return len(s) != len(set(s))
```

This function converts the string to a set (which removes duplicates) and compares lengths. If they differ, duplicates exist.

**Example usage:**
```python
has_duplicates("hello")    # True (l appears twice)
has_duplicates("world")    # False
has_duplicates("aabbcc")   # True
has_duplicates("abcdef")   # False
```


### Temperature

In [37]:
messages = []

add_user_message(messages,"Generate a movie idea in one sentence.")
text = chat(messages,temperature=0.0)

print(text)

A retired astronaut discovers her dreams are actually memories from a parallel universe where she never came home, and now both versions of her life are colliding.


### Streaming

In [41]:
messages = []

add_user_message(messages,"Write a 1 sentence description of a fake database")

response = client.converse_stream(messages=messages, modelId=model_id)

text = ""
for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]["delta"]["text"]
        print(chunk, end="")
        text += chunk

print("\n\nTotal Message: \n" + text)

A fake database is a simulated or mock data storage system used for testing, development, or demonstration purposes that mimics the structure and behavior of a real database without containing actual production data.

Total Message: 
A fake database is a simulated or mock data storage system used for testing, development, or demonstration purposes that mimics the structure and behavior of a real database without containing actual production data.


### Controlling User output

In [43]:
messages = []

add_user_message(messages, "Is coffee or tea better for breakfast?")
add_assistant_message(messages,"Coffee is better because")

chat(messages)

" it has more caffeine to wake you up quickly, but tea is better because it's gentler on your stomach and provides steadier energy.\n\n**Really, it depends on what you need:**\n\n- **Choose coffee if** you want a strong caffeine kick, enjoy bold flavors, or have a long morning ahead\n- **Choose tea if** you're sensitive to caffeine, want hydration with antioxidants, or prefer a calmer start\n\nBoth are healthy in moderation. Some people even enjoy both - tea for gentle waking, coffee mid-morning for a boost. What matters most for your morning routine?"

### Stop Sequences

In [46]:
messages = []

add_user_message(messages, "Count from 1 to 10")

chat(messages,stop_sequences=["5"])

'1, 2, 3, 4, '

### Structured data

In [51]:
messages = []

add_user_message(messages,"Generate a very sgort event bridge rule as json")

add_assistant_message(messages, "```json")

text = chat(messages,stop_sequences=["```"])

import json

json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

Exercise

In [54]:
messages = []

prompt = """Generate three different sample AWS CLI commands. Each should be very short"""

add_user_message(messages,prompt)

add_assistant_message(messages,"```bash")

text = chat(messages,stop_sequences=["```"])
print(text)




aws s3 ls

aws ec2 describe-instances

aws iam list-users

